In [3]:
import numpy as np 
import soundfile as sf
import librosa
import pandas as pd 
from pathlib import Path
import pickle 
import IPython.display as ipd
import sys 
sys.path.append('../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 
import soxr
import h5py
from tqdm.auto import tqdm
%matplotlib inline 
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore') # ignore pysound warning from librosa 

# Make stimuli for model simulation of 2024 mono word recognition experiment

### Same as human experiment, now using every possible foreground background pairing of stimuli - reflects full set presented to participants 
Using sanity-checked examples - removed stimuli with mismatched cue and target, or noisy excerpts. 

## Wanted Conditions:
### SNRs:
-9, -6, -3, 0, 3, inf 
### Distractor conditions:
- English (sex balanced: 2 same, 2 different per SNR )
- Mandarin (sex balanced: 2 same, 2 different per SNR )
- Dutch (sex balanced: 2 same, 2 different per SNR )
- 2-distractor
- 4-distractor
- 8-talker babble 
- Speech Shaped Noise
- Natural scenes 
- Music


Will be using cues, foregrounds, and english distractors for 1, 2, and 4 distractor cases taken from manifest:
`/om/user/imgriff/datasets/human_word_rec_SWC_2024/full_cue_target_distractor_df_w_meta_transcripts.pdpkl`
* for 4 distractor case, combine sets of 2 distractors (1 male set 1 female set = 2 of each = 4 total)

This manifest and
stimuli will be moved to `/om/user/imgriff/datasets/human_word_rec_SWC_2024`



In [4]:
# manifest = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/screened_eval_trial_manifest_new_fnames_w_transcripts.pdpkl')
# manifest.shape

In [5]:
stim_out_path = Path('/om/user/imgriff/datasets/human_word_rec_SWC_2024')
manifest = pd.read_pickle(stim_out_path/ 'human_cue_target_distractor_df_w_meta_transcripts_w_f0.pdpkl')


## We will store target and distractor excerpts separately in the same h5 file

Store cues, foregrounds, and distractors as separate datagroups in the same file - will select the distractor key based on the test. 
Allows for SNR combination on the fly 

In [38]:
util_audio.get_excerpt

<function util_audio.get_excerpt(dfi, dur=3.0, sr=50000, pad_with_context=True, jitter_fraction=0)>

In [39]:
manifest.columns

Index(['manifest_ix', 'orig_df_ix', 'client_id', 'corpus', 'gender',
       'gender_int', 'sr', 'word', 'cue_client_id', 'cue_corpus', 'cue_gender',
       'cue_gender_int', 'cue_sr', 'cue_total_file_duration_in_s', 'cue_word',
       'sex_cond', 'excerpt_src_fn', 'excerpt_cue_src_fn',
       'excerpt_same_sex_distractor_1_src_fn',
       'excerpt_same_sex_distractor_2_src_fn', 'same_sex_distractor_1_src_fn',
       'same_sex_distractor_2_src_fn', 'target_transcripts',
       'same_sex_distractor_1_transcripts',
       'same_sex_distractor_2_transcripts', 'same_sex_dist_1_word',
       'same_sex_dist_2_word', 'excerpt_diff_sex_distractor_1_src_fn',
       'excerpt_diff_sex_distractor_2_src_fn', 'diff_sex_distractor_1_src_fn',
       'diff_sex_distractor_2_src_fn', 'diff_sex_distractor_1_transcripts',
       'diff_sex_distractor_2_transcripts', 'diff_sex_dist_1_word',
       'diff_sex_dist_2_word', 'same_sex_zh_distractor_full_path',
       'diff_sex_zh_distractor_full_path', 'same_sex_

In [40]:
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))
manifest['word_int_label'] = manifest['word'].replace(word_dict)
manifest['same_dist_1_word_int'] = manifest['same_sex_dist_1_word'].replace(word_dict)
manifest['same_dist_2_word_int'] = manifest['same_sex_dist_2_word'].replace(word_dict)
manifest['diff_dist_1_word_int'] = manifest['diff_sex_dist_1_word'].replace(word_dict)
manifest['diff_dist_2_word_int'] = manifest['diff_sex_dist_2_word'].replace(word_dict)

In [41]:
def get_avg_f0(sig, SR, fmin=80, fmax=300):
    # get talker f0 - bound to range of human voice 
    target_f0, voice_flag, voice_probs = librosa.pyin(sig, fmin=80, fmax=300, sr=SR, fill_na=np.nan)
    target_f0 = target_f0[voice_flag]        
    avg_target_f0 = np.nanmean(target_f0)
    return avg_target_f0

In [42]:
## Set up parameters of dataset 

SR = 44100
dBSPL = 60 
# timing in seconds 
signal_dur = 2.5 # add context to crop to 2s post cochleagram 
win_dur = 10 # ms 

### init output h5 file 

output_file_name = stim_out_path / 'model_eval_stim.h5'

# init output key list
signal_keys  = ['target_signal',
              'cue_signal',
              "1-talker-english-same",
              "1-talker-english-different",
              "1-talker-mandarin-same",
              "1-talker-mandarin-different",
              "1-talker-dutch-same",
              "1-talker-dutch-different",
              "2-talker",
              "4-talker",
              'babble_dist_signal',
              'music_dist_signal',
              'nat_dist_signal',
              'ssn_dist_signal',
              ]

## Will only use dist_1 for same/different single english distractor cases 
label_keys = ['word_int_label', 'same_dist_word_int', 'diff_dist_word_int']

signal_len = int(SR * signal_dur)
n_examples = len(manifest)

In [43]:
output_file_name

PosixPath('/om/user/imgriff/datasets/human_word_rec_SWC_2024/model_eval_stim.h5')

In [44]:
print(manifest.columns)

Index(['manifest_ix', 'orig_df_ix', 'client_id', 'corpus', 'gender',
       'gender_int', 'sr', 'word', 'cue_client_id', 'cue_corpus', 'cue_gender',
       'cue_gender_int', 'cue_sr', 'cue_total_file_duration_in_s', 'cue_word',
       'sex_cond', 'excerpt_src_fn', 'excerpt_cue_src_fn',
       'excerpt_same_sex_distractor_1_src_fn',
       'excerpt_same_sex_distractor_2_src_fn', 'same_sex_distractor_1_src_fn',
       'same_sex_distractor_2_src_fn', 'target_transcripts',
       'same_sex_distractor_1_transcripts',
       'same_sex_distractor_2_transcripts', 'same_sex_dist_1_word',
       'same_sex_dist_2_word', 'excerpt_diff_sex_distractor_1_src_fn',
       'excerpt_diff_sex_distractor_2_src_fn', 'diff_sex_distractor_1_src_fn',
       'diff_sex_distractor_2_src_fn', 'diff_sex_distractor_1_transcripts',
       'diff_sex_distractor_2_transcripts', 'diff_sex_dist_1_word',
       'diff_sex_dist_2_word', 'same_sex_zh_distractor_full_path',
       'diff_sex_zh_distractor_full_path', 'same_sex_

In [45]:
def load_and_set_dbspl(file_path, sr, dbspl):
    """
    Load an audio file and set its dB SPL.
    
    Parameters:
    - file_path: Path to the audio file.
    - sr: Sampling rate to use for loading the audio.
    - dbspl: Desired dB SPL level.
    
    Returns:
    - Audio signal with the specified dB SPL.
    """
    audio_signal, _ = librosa.load(file_path, sr=sr)
    return util_audio.set_dBSPL(audio_signal, dbspl)


In [46]:
output_file_name

PosixPath('/om/user/imgriff/datasets/human_word_rec_SWC_2024/model_eval_stim.h5')

In [47]:
np.random.seed(0)


with h5py.File(output_file_name, 'w') as f:
    for key in signal_keys:
        f.create_dataset(key, shape=[n_examples, signal_len], dtype=np.float32)
    for key in label_keys:
        f.create_dataset(key, shape=[n_examples], dtype=np.int32)
    # Read, cut and save the cue, target and distractor signals
    for ix, row in enumerate(tqdm(manifest.itertuples(), total=n_examples)):
        # save labels 
        f['word_int_label'][ix] = row.word_int_label
        # get target signal
        cue_signal = load_and_set_dbspl(row.excerpt_cue_src_fn, SR, dBSPL)
        target_signal = load_and_set_dbspl(row.excerpt_src_fn, SR, dBSPL)
        f['cue_signal'][ix] = util_audio.pad_or_trim_to_len(cue_signal, signal_len, mode='both')
        f['target_signal'][ix] = util_audio.pad_or_trim_to_len(target_signal, signal_len, mode='both')
        
        ## Get english distractors 
        # same sex 
        same_eng_1_signal = load_and_set_dbspl(row.excerpt_same_sex_distractor_1_src_fn, SR, dBSPL)
        f['1-talker-english-same'][ix] = util_audio.pad_or_trim_to_len(same_eng_1_signal, signal_len, mode='both')
        f['same_dist_word_int'][ix] = row.same_dist_1_word_int
        # diff sex 
        diff_eng_1_signal = load_and_set_dbspl(row.excerpt_diff_sex_distractor_1_src_fn, SR, dBSPL)
        f['1-talker-english-different'][ix] = util_audio.pad_or_trim_to_len(diff_eng_1_signal, signal_len, mode='both')
        f['diff_dist_word_int'][ix] = row.diff_dist_1_word_int

        ## Get 2-talker distractors
        two_talker_signal = util_audio.combine_signal_and_noise(same_eng_1_signal, diff_eng_1_signal, 0)
        f['2-talker'][ix] = util_audio.pad_or_trim_to_len(two_talker_signal, signal_len, mode='both')

        ## Get 4-talker distractors
        same_eng_2_signal = load_and_set_dbspl(row.excerpt_same_sex_distractor_2_src_fn, SR, dBSPL)
        diff_eng_2_signal = load_and_set_dbspl(row.excerpt_diff_sex_distractor_2_src_fn, SR, dBSPL)
        four_dist_signal = np.sum([same_eng_1_signal, same_eng_2_signal, diff_eng_1_signal, diff_eng_2_signal], axis=0)
        f['4-talker'][ix] = util_audio.pad_or_trim_to_len(four_dist_signal, signal_len, mode='both')

        ## Get mandarin distractors
        # same sex 
        same_mand_signal = load_and_set_dbspl(row.same_sex_zh_distractor_full_path, SR, dBSPL)
        f['1-talker-mandarin-same'][ix] = util_audio.pad_or_trim_to_len(same_mand_signal, signal_len, mode='both')
        # different
        diff_mand_signal = load_and_set_dbspl(row.diff_sex_zh_distractor_full_path, SR, dBSPL)
        f['1-talker-mandarin-different'][ix] = util_audio.pad_or_trim_to_len(diff_mand_signal, signal_len, mode='both')

        ## Get dutch distractors 
        # same
        same_dutch_signal = load_and_set_dbspl(row.same_sex_nl_distractor_full_path, SR, dBSPL)
        f['1-talker-dutch-same'][ix] = util_audio.pad_or_trim_to_len(same_dutch_signal, signal_len, mode='both')
        # different
        diff_dutch_signal = load_and_set_dbspl(row.diff_sex_nl_distractor_full_path, SR, dBSPL)
        f['1-talker-dutch-different'][ix] = util_audio.pad_or_trim_to_len(diff_dutch_signal, signal_len, mode='both')

        # gen ssn distractor
        ssn_signal = util_audio.spectrally_matched_noise(target_signal, SR)
        f['ssn_dist_signal'][ix] = util_audio.pad_or_trim_to_len(
                                        util_audio.set_dBSPL(ssn_signal, dBSPL),
                                        signal_len, mode='both')

        # get music distractor
        music_signal = load_and_set_dbspl(row.music_full_path, SR, dBSPL)
        f['music_dist_signal'][ix] = util_audio.pad_or_trim_to_len(music_signal, signal_len, mode='both')

        # get babble distractor
        babble_signal = load_and_set_dbspl(row.babble_full_path, SR, dBSPL)
        f['babble_dist_signal'][ix] = util_audio.pad_or_trim_to_len(babble_signal, signal_len, mode='both')

        # get natural scene distractor
        nat_signal = load_and_set_dbspl(row.natural_scene_full_path, SR, dBSPL)
        f['nat_dist_signal'][ix] = util_audio.pad_or_trim_to_len(nat_signal, signal_len, mode='both')
        

        





  0%|          | 0/976 [00:00<?, ?it/s]

In [ ]:
# distractor_eg.distractor_word

'studio'

In [49]:
## Check file 
### init output h5 file 
stim_out_path = Path('/om/user/imgriff/datasets/human_word_rec_SWC_2024/')

manifest = pd.read_pickle(stim_out_path/ 'human_cue_target_distractor_df_w_meta_transcripts_w_f0.pdpkl')


output_file_name = stim_out_path / 'model_eval_stim.h5'
h5_file = h5py.File(output_file_name, 'r')

In [52]:
for key in h5_file.keys():
    print(key, h5_file[key].shape)

1-talker-dutch-different (976, 110250)
1-talker-dutch-same (976, 110250)
1-talker-english-different (976, 110250)
1-talker-english-same (976, 110250)
1-talker-mandarin-different (976, 110250)
1-talker-mandarin-same (976, 110250)
2-talker (976, 110250)
4-talker (976, 110250)
babble_dist_signal (976, 110250)
cue_signal (976, 110250)
diff_dist_word_int (976,)
music_dist_signal (976, 110250)
nat_dist_signal (976, 110250)
same_dist_word_int (976,)
ssn_dist_signal (976, 110250)
target_signal (976, 110250)
word_int_label (976,)


In [51]:
for key in h5_file.keys():
    if 'f0' in key:
        print(key, h5_file[key].shape)
        data = h5_file[key][:]
        ixs = np.where(np.logical_or(data > 300 , data < 80))[0]
        if len(ixs) > 0:
            print(key)


In [20]:
# Add f0s to manifest and save 
manifest['target_f0'] = h5_file['target_f0'][:]
manifest['cue_f0'] = h5_file['cue_f0'][:]
manifest['distractor_f0'] = h5_file['one_dist_f0'][:]

In [22]:
h5_file.close()

In [23]:
manifest_out_name = "/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/screened_eval_trial_manifest_new_fnames_w_transcripts_and_f0.pdpkl"
manifest.to_pickle(manifest_out_name)

### Dev dataset class

In [14]:
import importlib 
import corpus.swc_mono_test as swc_corpus
importlib.reload(swc_corpus)
output_file_name = stim_out_path / 'model_eval_stim.h5'

keys  = [
              "1-talker-english-same",
              "1-talker-english-different",
              "1-talker-mandarin-same",
              "1-talker-mandarin-different",
              "1-talker-dutch-same",
              "1-talker-dutch-different",
              "2-talker",
              "4-talker",
              'babble',
              'music',
              'natural_scene',
              'stationary',
              ]
    
for key in keys:
    print(key)
    dataset = swc_corpus.SWCMonoTestSetH5Dataset(output_file_name, key, 44_100)
    for _ in dataset:
        continue 

1-talker-english-same
False
same_dist_word_int
same_dist_word_int


  0%|          | 0/976 [00:00<?, ?it/s]

1-talker-english-different
False
diff_dist_word_int
diff_dist_word_int


  0%|          | 0/976 [00:00<?, ?it/s]

1-talker-mandarin-same
True
None
same_dist_word_int


...

  0%|          | 0/976 [00:00<?, ?it/s]

natural_scene
True
None
same_dist_word_int


  0%|          | 0/976 [00:00<?, ?it/s]

stationary
True
None
same_dist_word_int


  0%|          | 0/976 [00:00<?, ?it/s]

In [20]:
"2024" in str(output_file_name)

True